## Setup: the imports

Each line brings in a tool under a short name.

- `import pandas as pd`, `numpy as np`, `matplotlib.pyplot as plt`: load the three libraries with nicknames, so we type `pd` instead of `pandas`.
- `from sklearn.model_selection import train_test_split`: the function that splits data into a training part and a test part.
- `from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree`: the tree model plus two ways to view a tree.
- `from sklearn.naive_bayes import CategoricalNB, GaussianNB`: the two Naive Bayes variants (categorical features vs numeric).
- `from sklearn.metrics import ...`: the scoring tools (accuracy, confusion matrix, full report).
- `pd.set_option("display.max_columns", None)`: a display setting so pandas never hides columns when printing a wide table.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

ETSU_NAVY = "#00053E"
ETSU_GOLD = "#FFC72C"
pd.set_option("display.max_columns", None)
print("Libraries loaded.")

Libraries loaded.


# Step 1: from the lecture (classification lecture example) to code

## Building the dataframe.
#### You can also create one in a csv file and use that instead of building this dataframe

- `{ "age": [...], ... }`: a Python **dictionary**. Each key is a column name; each value is a list of that column's values going down the rows.
- `pd.DataFrame(data)`: turns that dictionary into a table (a **DataFrame**).

In [2]:
data = {
    "age":          ["<=30","<=30","31-40",">40",">40",">40","31-40","<=30","<=30",">40","<=30","31-40","31-40",">40"],
    "income":       ["high","high","high","medium","low","low","low","medium","low","medium","medium","medium","high","medium"],
    "student":      ["no","no","no","no","yes","yes","yes","no","yes","yes","yes","no","yes","no"],
    "credit_rating":["fair","excellent","fair","fair","fair","excellent","excellent","fair","fair","fair","excellent","excellent","fair","excellent"],
    "buys_computer":["no","no","yes","yes","yes","no","yes","no","yes","yes","yes","yes","yes","no"],
}
buys = pd.DataFrame(data)
buys

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes
5,>40,low,yes,excellent,no
6,31-40,low,yes,excellent,yes
7,<=30,medium,no,fair,no
8,<=30,low,yes,fair,yes
9,>40,medium,yes,fair,yes


### Step 2: Encoding the column values in our dataset 

In [3]:
maps = {
    "age":           {"<=30": 0, "31-40": 1, ">40": 2},
    "income":        {"low": 0, "medium": 1, "high": 2},
    "student":       {"no": 0, "yes": 1},
    "credit_rating": {"fair": 0, "excellent": 1},
}
X = buys[list(maps)].apply(lambda col: col.map(maps[col.name]))
y = buys["buys_computer"]
X

,age,income,student,credit_rating
0,0,2,0,0
1,0,2,0,1
2,1,2,0,0
3,2,1,0,0
4,2,0,1,0
5,2,0,1,1
6,1,0,1,1
7,0,1,0,0
8,0,0,1,0
9,2,1,1,0


**Look closer at the lambda.** Let's unpack that one line by running its pieces.

In [4]:
# 1) list() on a dictionary gives its keys
print("list(maps) =", list(maps))

# 2) .map on ONE column swaps each value using that column's dictionary
print("\nThe age column, encoded by itself:")
print(buys["age"].map(maps["age"]).head().to_string())

# 3) .apply runs that same idea on every column at once.
#    Inside the lambda, 'col' is each column; col.name tells us which dictionary to use.
for name in list(maps):
    column = buys[name]
    print(f"  column handed in -> col.name = '{column.name}', so .apply uses maps['{column.name}']")

list(maps) = ['age', 'income', 'student', 'credit_rating']

The age column, encoded by itself:
0    0
1    0
2    1
3    2
4    2
  column handed in -> col.name = 'age', so .apply uses maps['age']
  column handed in -> col.name = 'income', so .apply uses maps['income']
  column handed in -> col.name = 'student', so .apply uses maps['student']
  column handed in -> col.name = 'credit_rating', so .apply uses maps['credit_rating']


### Step 3: Train the decision tree

- `DecisionTreeClassifier(criterion="gini", random_state=0)`: creates the model. `criterion="gini"` splits using Gini impurity (the measure from lecture). `random_state=0` fixes the randomness so the tree is the same every run.
- `.fit(X, y)`: **trains** the model, learning the splits from features and answers. The single most important method in sklearn.
- `export_text(tree, feature_names=list(maps))`: prints the trained tree as indented text. `feature_names=` shows real column names instead of "feature 0."
- `print(...)`: writes it to the screen.

Notice the tree roots on `student`, not `age` like the hand tree. That is because sklearn
uses binary yes/no splits (the CART algorithm) while we used multi-way information gain.
Different recipe, same patterns, and as the next cell shows, the same prediction.

In [5]:
tree = DecisionTreeClassifier(criterion="gini", random_state=0)
tree.fit(X, y)
print(export_text(tree, feature_names=list(maps)))

|--- student <= 0.50
|   |--- age <= 0.50
|   |   |--- class: no
|   |--- age >  0.50
|   |   |--- age <= 1.50
|   |   |   |--- class: yes
|   |   |--- age >  1.50
|   |   |   |--- credit_rating <= 0.50
|   |   |   |   |--- class: yes
|   |   |   |--- credit_rating >  0.50
|   |   |   |   |--- class: no
|--- student >  0.50
|   |--- credit_rating <= 0.50
|   |   |--- class: yes
|   |--- credit_rating >  0.50
|   |   |--- age <= 1.50
|   |   |   |--- class: yes
|   |   |--- age >  1.50
|   |   |   |--- class: no



### Step 4: Predict the instance we did by hand

- `pd.DataFrame([{...}])`: a list holding one dictionary becomes a **one-row** DataFrame (our test person, already encoded).
- `tree.predict(instance)`: runs the person down the tree and returns an **array** of predictions.
- `[0]`: array indexing. `predict` always returns an array, so `[0]` pulls out the first and only answer.

In [6]:
instance = pd.DataFrame([{"age": 0, "income": 1, "student": 1, "credit_rating": 0}])
print("Decision tree prediction:", tree.predict(instance)[0])

Decision tree prediction: yes


### Step 5: Train a Naive Bayes classifier on the same instance

- `CategoricalNB()`: creates categorical Naive Bayes (defaults are fine).
- `.fit(X, y)`: trains it, same method as the tr

In [ ]:
nb = CategoricalNB()
nb.fit(X, y)

Naive Bayes prediction: yes
  P(no) = 0.232
  P(yes) = 0.768


### Step 6: Predict the instance using Naive Bays classifier
- `.predict(instance)[0]`: the single predicted label.
- `.predict_proba(instance)`: returns the **probability** of each class as a 2D array (one row per input); `[0]` takes our one row.
- `nb.classes_`: an **attribute** (no parentheses: stored data, not an action) listing the class labels in the order the probabilities come in.
- `zip(nb.classes_, proba)`: pairs each label with its probability so we can loop over both together.
- `f"P({label}) = {p:.3f}"`: an **f-string**; `{...}` inserts a value and `:.3f` shows 3 decimals.

In [ ]:
print("Naive Bayes prediction:", nb.predict(instance)[0])

proba = nb.predict_proba(instance)[0]
for label, p in zip(nb.classes_, proba):
    print(f"  P({label}) = {p:.3f}")